# UGTM Plagiarism AI — Remote GPU Inference Worker

Notebook này thiết lập một dịch vụ suy luận GPU từ xa trên hạ tầng Google Colab (NVIDIA Tesla T4/L4), đảm nhận việc tính toán hai mô hình học sâu:
1. **Dense Vector Embedding:** `BAAI/bge-m3` (1024 chiều)
2. **Cross-Encoder Reranking:** `BAAI/bge-reranker-v2-m3`

### Quy trình vận hành:
1. Chọn **Runtime** → **Change runtime type** → **Hardware accelerator: T4 GPU** (hoặc L4 nếu có Colab Pro) → **Save**.
2. (Tùy chọn bảo mật) Điền khóa `GPU_WORKER_API_KEY` ở Khối 3 nếu muốn bảo vệ endpoint bằng token.
3. Thực thi toàn bộ các khối lệnh bên dưới (**Runtime** → **Run all**).
4. Sao chép địa chỉ `REMOTE_GPU_URL` (và `REMOTE_GPU_API_KEY` nếu có) xuất hiện ở cell cuối cùng và cập nhật vào file cấu hình `.env` của hệ thống máy chủ cục bộ / Render.

In [ ]:
# 1. Cài đặt các gói thư viện phụ thuộc (khớp requirements/gpu-colab.txt)
!pip install -q "torch>=2.6.0" "safetensors>=0.4.5" "FlagEmbedding>=1.3.4" "transformers>=4.48,<5" fastapi uvicorn pycloudflared nest-asyncio requests

In [ ]:
# 2. Kiểm tra tài nguyên phần cứng GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA device not detected. Please select Runtime -> Change runtime type -> T4 GPU.")

device_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
compute_cap = torch.cuda.get_device_capability(0)

print(f"Device Name       : {device_name}")
print(f"Compute Capability: {compute_cap[0]}.{compute_cap[1]}")
print(f"Total VRAM        : {total_vram_gb:.2f} GB")
print(f"CUDA Version      : {torch.version.cuda}")
print(f"PyTorch Version   : {torch.__version__}")

In [ ]:
# 3. Nạp mô hình học sâu lên bộ nhớ GPU (FP16 & Ghim Revision)
import logging
import os
from FlagEmbedding import BGEM3FlagModel, FlagReranker

logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("ugtm_gpu_worker")

# Khóa phiên bản và định danh model
EMBEDDING_MODEL = "BAAI/bge-m3"
EMBEDDING_REVISION = os.environ.get("EMBEDDING_REVISION", "main")
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
RERANKER_REVISION = os.environ.get("RERANKER_REVISION", "main")

# Token bảo vệ endpoint (đặt chuỗi ngẫu nhiên hoặc để trống nếu chỉ test nhanh)
GPU_WORKER_API_KEY = os.environ.get("GPU_WORKER_API_KEY", "").strip()

logger.info("Loading embedding model: %s (rev=%s) onto CUDA...", EMBEDDING_MODEL, EMBEDDING_REVISION)
embedder = BGEM3FlagModel(
    EMBEDDING_MODEL,
    use_fp16=True,
    devices=["cuda:0"],
)
logger.info("Embedding model %s initialized successfully.", EMBEDDING_MODEL)

logger.info("Loading reranker model: %s (rev=%s) onto CUDA...", RERANKER_MODEL, RERANKER_REVISION)
reranker = FlagReranker(
    RERANKER_MODEL,
    use_fp16=True,
    devices=["cuda:0"],
)
logger.info("Reranker model %s initialized successfully.", RERANKER_MODEL)

In [ ]:
# 4. Khởi tạo REST API Chuẩn hóa và Mở cổng kết nối (Cloudflare Tunnel)
import secrets
import threading
import time
from typing import List, Optional
from fastapi import FastAPI, Header, HTTPException, Security
from pydantic import BaseModel, Field
import uvicorn
from pycloudflared import try_cloudflare

app = FastAPI(
    title="UGTM Remote GPU Inference Worker",
    version="1.1.0",
    description="Dedicated GPU inference endpoint for UGTM Plagiarism Engine",
)

def verify_auth_token(
    authorization: Optional[str] = Header(None),
    x_api_key: Optional[str] = Header(None),
):
    if not GPU_WORKER_API_KEY:
        return
    token = None
    if authorization and authorization.startswith("Bearer "):
        token = authorization.split("Bearer ", 1)[1].strip()
    elif x_api_key:
        token = x_api_key.strip()
    if not token or not secrets.compare_digest(token, GPU_WORKER_API_KEY):
        raise HTTPException(status_code=401, detail="Unauthorized GPU Worker Token")

class EncodeRequest(BaseModel):
    texts: List[str] = Field(..., description="List of input passage chunks to vectorize")
    batch_size: int = Field(default=32, ge=1, le=128)

class RerankRequest(BaseModel):
    pairs: List[List[str]] = Field(..., description="List of candidate pairs [query_text, source_text]")
    batch_size: int = Field(default=32, ge=1, le=128)

@app.get("/health")
def health_check(auth=Security(verify_auth_token)):
    return {
        "status": "healthy",
        "device": torch.cuda.get_device_name(0),
        "cuda": True,
        "vram_allocated_mb": round(torch.cuda.memory_allocated(0) / (1024**2), 2),
        "embedding_model": EMBEDDING_MODEL,
        "embedding_revision": EMBEDDING_REVISION,
        "embedding_dim": 1024,
        "reranker_model": RERANKER_MODEL,
        "reranker_revision": RERANKER_REVISION,
    }

@app.post("/encode")
def encode_endpoint(payload: EncodeRequest, auth=Security(verify_auth_token)):
    if len(payload.texts) > 128:
        raise HTTPException(status_code=413, detail="Batch size exceeded cap (max 128 passages per request)")
    start_time = time.time()
    output = embedder.encode(
        payload.texts,
        batch_size=payload.batch_size,
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    dense_vecs = output["dense_vecs"].tolist()
    duration = time.time() - start_time
    throughput = len(payload.texts) / max(duration, 0.001)
    logger.info("Encoded %d passages in %.2fs (%.1f passages/s)", len(payload.texts), duration, throughput)
    return {
        "model": EMBEDDING_MODEL,
        "revision": EMBEDDING_REVISION,
        "dimension": 1024,
        "count": len(dense_vecs),
        "dense_vecs": dense_vecs,
    }

@app.post("/rerank")
def rerank_endpoint(payload: RerankRequest, auth=Security(verify_auth_token)):
    if len(payload.pairs) > 128:
        raise HTTPException(status_code=413, detail="Batch size exceeded cap (max 128 pairs per request)")
    start_time = time.time()
    scores = reranker.compute_score(
        payload.pairs,
        batch_size=payload.batch_size,
        normalize=True,
        max_length=512,
    )
    duration = time.time() - start_time
    if isinstance(scores, (float, int)):
        normalized_scores = [float(scores)]
    else:
        normalized_scores = [float(s) for s in scores]
    logger.info("Reranked %d candidate pairs in %.2fs", len(payload.pairs), duration)
    return {
        "model": RERANKER_MODEL,
        "revision": RERANKER_REVISION,
        "count": len(normalized_scores),
        "scores": normalized_scores,
    }

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

tunnel_instance = try_cloudflare(port=8000)
endpoint_url = tunnel_instance.tunnel

separator = "=" * 72
print("\n" + separator)
print("UGTM REMOTE GPU INFERENCE WORKER — ONLINE")
print(separator)
print(f"Target Hardware   : {device_name} ({total_vram_gb:.1f} GB VRAM)")
print(f"Public Endpoint   : {endpoint_url}")
if GPU_WORKER_API_KEY:
    print("Auth Token Status : CONFIGURED (Authentication Required)")
else:
    print("Auth Token Status : OPEN (No token required)")
print("\nSystem Configuration:")
print("Add or update the following entry in your local .env configuration:")
print(f"REMOTE_GPU_URL={endpoint_url}")
if GPU_WORKER_API_KEY:
    print(f"REMOTE_GPU_API_KEY={GPU_WORKER_API_KEY}")
print(separator + "\n")

while True:
    time.sleep(60)